In [64]:
!pip install -q cassio datasets langchain openai tiktoken

In [65]:
# LangChain components to use
from langchain.vectorstores.cassandra import Cassandra
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.indexes.vectorstore import VectorStoreIndexWrapper
from langchain.llms import OpenAI

#Support for dataset retrieval with Hugging Face Datasets
from datasets import load_dataset

# With CassIO, the engine powering the Astra DB Integration in LangChain,
# you will also initialize the connection to your Astra DB instance.
import cassio


In [66]:
!pip install PyPDF2


In [67]:
from PyPDF2 import PdfReader

In [68]:
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Access the variables
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_ID = os.getenv("ASTRA_DB_ID")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ASTRA_DB_KEYSPACE = os.getenv("ASTRA_DB_KEYSPACE")

In [69]:
# provide the path to the PDF file
pdfreader = PdfReader("documents/budget_speech.pdf")

In [70]:
from typing_extensions import Concatenate

# Extract text from each page of the PDF
raw_text = ""
for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text += content + "\n"
    else:
        print(f"Page {i} is empty or could not be read.")

Page 1 is empty or could not be read.
Page 3 is empty or could not be read.


In [71]:
raw_text

'GOVERNMENT OF INDIA\nBUDGET 2025-2026\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2025\n \nCONTENTS  \n \nPART – A \n Page No.  \nIntroduction  1 \nBudget Theme  1 \nAgriculture as the 1st engine  3 \nMSMEs as the 2nd engine  6 \nInvestment as the 3rd engine  8 \nA. Investing in People  8 \nB. Investing in  the Economy  10 \nC. Investing in Innovation  14 \nExports as the 4th engine  15 \nReforms as the Fuel  16 \nFiscal Policy  18 \n \n \nPART – B \nIndirect taxes  20 \nDirect Taxes   23 \n \nAnnexure to Part -A 29 \nAnnexure to Part -B 31 \n \n  \n \n \nBudget 202 5-2026 \n \nSpeech of  \nNirmala Sitharaman  \nMinister of Finance  \nFebruary 1 , 202 5 \nHon’ble Speaker,  \n I present the Budget for 2025 -26. \nIntroduction  \n1. This Budget continues our Government ’s efforts to:  \na) accelerate growth,  \nb) secure inclusive development,  \nc) invigorate private sector investments,  \nd) uplift household sentiments, and \ne) enhance spending power of India’s

In [72]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple whitespace
    return text.strip()

raw_text = clean_text(raw_text)


In [73]:

cassio.init(
    database_id=ASTRA_DB_ID,
    token=ASTRA_DB_APPLICATION_TOKEN,
    keyspace=ASTRA_DB_KEYSPACE
)


In [74]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Use ChatOpenAI for chat models like gpt-3.5-turbo
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=OPENAI_API_KEY
)

# This part looks correct
embeddings = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY
)

In [75]:
astra_vector_store = Cassandra(
    embedding=embeddings,
    table_name="langchain_vector_store",
    session=None,  # CassIO handles the session automatically after cassio.init()
    keyspace=ASTRA_DB_KEYSPACE,
)

In [76]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
texts = []

for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        cleaned = clean_text(content)
        chunks = splitter.split_text(cleaned)
        for chunk in chunks:
            texts.append({
                "page_content": chunk,
                "metadata": {"page_number": i + 1}
            })


In [77]:
texts[:50]  # Display the first 50 chunks to verify the split

[{'page_content': 'GOVERNMENT OF INDIA BUDGET 2025-2026 SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE February 1, 2025',
  'metadata': {'page_number': 1}},
 {'page_content': 'CONTENTS PART – A Page No. Introduction 1 Budget Theme 1 Agriculture as the 1st engine 3 MSMEs as the 2nd engine 6 Investment as the 3rd engine 8 A. Investing in People 8 B. Investing in the Economy 10 C. Investing in Innovation 14 Exports as the 4th engine 15 Reforms as the Fuel 16 Fiscal Policy 18 PART – B Indirect taxes 20 Direct Taxes 23 Annexure to Part -A 29 Annexure to Part -B 31',
  'metadata': {'page_number': 3}},
 {'page_content': 'Budget 202 5-2026 Speech of Nirmala Sitharaman Minister of Finance February 1 , 202 5 Hon’ble Speaker, I present the Budget for 2025 -26. Introduction 1. This Budget continues our Government ’s efforts to: a) accelerate growth, b) secure inclusive development, c) invigorate private sector investments, d) uplift household sentiments, and e) enhance spending power of India’s 

In [78]:
budget_chunks = [text for text in texts if "budget" in text.lower()]
for chunk in budget_chunks:
    print("\n---\n", chunk[:500])  # Print a preview


AttributeError: 'dict' object has no attribute 'lower'

In [ ]:
astra_vector_store.add_texts(
    texts=[doc["page_content"] for doc in texts],
    metadatas=[doc["metadata"] for doc in texts]
)


['1ef9954ade224927b43f243b6e5214aa',
 '5032e59cddf04f1fac4fe39fe733d15f',
 '9ae620a6d6384383b8b125105f39aea2',
 'a74939ca8ae54828b7f2c57a2d167435',
 'ced3640cb2264161a87c9ff2c5c445bd',
 'ea81d5c5eb1e440a86afa272b0c100b0',
 'a5348c61b4704ff59837a552c0df542a',
 '928e18214a624a3c921ec461fa7766e4',
 '5b87f3b1fe13491d853e8b8b3000bfbf',
 '56ccdbd671d1476ea23c1fba9ca588ef',
 '31bbbff6621a40a19ea91746de0d8f5e',
 '0bc74a0387c048798f0dcd62408f58a2',
 '9063a289efcb488f94664f51e723b1ee',
 '960a50dade9b4e55b8d3a88de02c3dee',
 '508035f2508e4324a8bc98778f6e71a4',
 'd41446b661f049e39f03158046e73ea4',
 '8ad494084db4471eab490301bbf5831d',
 'b0619d5fc6684f6ba0bed85b768be08e',
 'fa51696ba81c4d10833a2daeca097a2c',
 '542e45fa0e9440b190ffe5da59747f90',
 '6e06edd8b8d54d6ba617957e676d88ff',
 '9bb656bf67854ee28aef48785a47e916',
 'da568a36b70a46989754f5ca342e08ef',
 '64b3d32e958c48d296d587060bfc928d',
 '0d8fa6b524c64e009368ac4983286d5d',
 '7e87f8ef9abc45bab5ee2261bbba8cf5',
 '876dbf9bae9d43bc88c2357ef140ce14',
 

In [ ]:
# 9. Setup RetrievalQA
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=astra_vector_store.as_retriever(),
    return_source_documents=True
)


In [80]:
# 10. Ask a specific budget-related query
query = """
List all major financial allocations in the budget speech along with the amount in ₹.
"""
response = qa.invoke({"query": query})

# 11. Print response and source info
print("Answer:", response["result"])
print("Source docs:")
for doc in response["source_documents"]:
    page = doc.metadata.get("page_number", "Unknown")
    print(f"Page: {page}, Content snippet: {doc.page_content[:200]}")

Answer: 1. Outlay of ₹1.5 lakh crore for 50-year interest-free loans to states for capital expenditure and incentives for reforms.
2. Asset Monetization Plan 2025-30 with a capital of ₹10 lakh crore for new projects.
3. Jal Jeevan Mission providing potable tap water connections to 15 crore households.
4. Total receipts other than borrowings amounting to ₹31.47 lakh crore.
5. Net tax receipts of ₹25.57 lakh crore.
6. Total expenditure of ₹47.16 lakh crore, with capital expenditure at about ₹10.18 lakh crore.
7. Revised Estimate of fiscal deficit at 4.8% of GDP.
Source docs:
Page: 32.0, Content snippet: (25% of his tax payable as per existing rates). 160. Details of my tax proposals are given in the Annexure. 161. As a result of these proposals, revenue of about ₹ 1 lakh crore in direct taxes and ₹ 2
Page: 14.0, Content snippet: 53. An outlay of ` 1.5 lakh crore is proposed for the 50 -year interest free loans to states for capital expenditure and incentives for reforms. Asset Monetizati